In [1]:
!git clone https://github.com/chufangao/CTOD.git
!git clone https://github.com/futianfan/clinical-trial-outcome-prediction.git
#!wget https://huggingface.co/datasets/chufangao/CTO/resolve/main/CTTI.zip
#CTTI_PATH = './CTTI.zip'


Cloning into 'CTOD'...
Cloning into 'clinical-trial-outcome-prediction'...
Updating files:  68% (81/119)
Updating files:  69% (83/119)
Updating files:  70% (84/119)
Updating files:  71% (85/119)
Updating files:  72% (86/119)
Updating files:  73% (87/119)
Updating files:  74% (89/119)
Updating files:  75% (90/119)
Updating files:  76% (91/119)
Updating files:  77% (92/119)
Updating files:  78% (93/119)
Updating files:  79% (95/119)
Updating files:  80% (96/119)
Updating files:  81% (97/119)
Updating files:  82% (98/119)
Updating files:  83% (99/119)
Updating files:  84% (100/119)
Updating files:  85% (102/119)
Updating files:  86% (103/119)
Updating files:  87% (104/119)
Updating files:  88% (105/119)
Updating files:  89% (106/119)
Updating files:  90% (108/119)
Updating files:  91% (109/119)
Updating files:  92% (110/119)
Updating files:  93% (111/119)
Updating files:  94% (112/119)
Updating files:  95% (114/119)
Updating files:  96% (115/119)
Updating files:  97% (116/119)
Updating fi

In [7]:
# # if you want to use the latest version of clinical trials instead, uncomment and run this cell
!pip install selenium
!python ./download_ctti.py
CTTI_PATH = './downloads/CTTI_new.zip'

python: can't open file 'C:\\Users\\Carol\\Documents\\Data Code and Deliverables\\Data Group Part\\download_ctti.py': [Errno 2] No such file or directory


In [8]:
import pandas as pd
# Load clinical trial outcome labels from CTO
CTO_phase1_preds = pd.read_csv("https://huggingface.co/datasets/chufangao/CTO/raw/main/phase1_CTO_rf.csv")[['nct_id', 'pred']].rename(columns={'pred': 'label'})
CTO_phase2_preds = pd.read_csv("https://huggingface.co/datasets/chufangao/CTO/raw/main/phase2_CTO_rf.csv")[['nct_id', 'pred']].rename(columns={'pred': 'label'})
CTO_phase3_preds = pd.read_csv("https://huggingface.co/datasets/chufangao/CTO/raw/main/phase3_CTO_rf.csv")[['nct_id', 'pred']].rename(columns={'pred': 'label'})

In [9]:
import zipfile
def load_all_studies_with_features( ctti_path = './downloads/CTTI_new.zip'):
    with zipfile.ZipFile(ctti_path, 'r') as zip_ref:
        names = zip_ref.namelist()
        studies = pd.read_csv(zip_ref.open([n for n in names if 'studies.txt' in n][0]), sep='|')
        diseases = pd.read_csv(zip_ref.open([n for n in names if 'browse_conditions.txt' in n][0]), sep='|')
        interventions = pd.read_csv(zip_ref.open([n for n in names if 'interventions.txt' in n][0]), sep='|')
        criteria = pd.read_csv(zip_ref.open([n for n in names if 'eligibilities.txt' in n][0]), sep='|')
        designs = pd.read_csv(zip_ref.open([n for n in names if 'designs.txt' in n][0]), sep='|')

    diseases = diseases.groupby('nct_id')['downcase_mesh_term'].apply(lambda x: '; '.join(x)).reset_index().rename(columns={'downcase_mesh_term': 'diseases'})
    interventions = interventions.dropna(subset=['downcase_mesh_term'])
    interventions['downcase_mesh_term'] = interventions['downcase_mesh_term'].str.lower()
    interventions = interventions.groupby('nct_id')['downcase_mesh_term'] \
        .apply(lambda x: '; '.join(sorted(set(x)))) \
        .reset_index().rename(columns={'downcase_mesh_term': 'drugs'})
    criteria = criteria.dropna(subset=['criteria']).drop_duplicates(subset=['nct_id'])
    criteria['criteria'] = criteria['criteria'].str.lower()
    designs['design'] = designs[['allocation', 'intervention_model', 'observational_model', 'primary_purpose', 'time_perspective', 'masking']].fillna('').agg(' '.join, axis=1).str.lower()
    designs = designs[['nct_id', 'design']].drop_duplicates()

    studies = studies.dropna(subset=['completion_date'])
    studies = studies.merge(diseases, on='nct_id', how='left') \
                     .merge(interventions, on='nct_id', how='left') \
                     .merge(criteria, on='nct_id', how='left') \
                     .merge(designs, on='nct_id', how='left')
    
    # Criar campo 'features' com todos os textos multimodais
    studies['features'] = studies[['phase', 'diseases', 'drugs', 'design', 'criteria']].fillna('').agg(' '.join, axis=1)
    return studies

# Altere este caminho se necessário:
studies_with_features = load_all_studies_with_features("./downloads/CTTI_new.zip")

C:\Users\Carol\AppData\Local\Temp\ipykernel_15164\150195257.py:5: DtypeWarning: Columns (47,48,53,68) have mixed types. Specify dtype option on import or set low_memory=False.
  studies = pd.read_csv(zip_ref.open([n for n in names if 'studies.txt' in n][0]), sep='|')


In [10]:
studies_with_features

,nct_id,nlm_download_date_description,study_first_submitted_date,results_first_submitted_date,disposition_first_submitted_date,last_update_submitted_date,study_first_submitted_qc_date,study_first_posted_date,study_first_posted_date_type,results_first_submitted_qc_date,...,healthy_volunteers,population,criteria,gender_description,gender_based,adult,child,older_adult,design,features
0,NCT04892017,NaN,2021-05-06,NaN,NaN,2025-01-16,2021-05-13,2021-05-19,ACTUAL,NaN,...,f,NaN,inclusion criteria:~1. male or female particip...,NaN,NaN,t,f,t,non_randomized sequential treatment none,PHASE1/PHASE2 neoplasms antineoplastic agents;...
1,NCT02598531,NaN,2015-10-23,NaN,NaN,2025-01-20,2015-11-04,2015-11-06,ESTIMATED,NaN,...,f,Study population is defined using the inclusio...,inclusion criteria:~* age 22-75 years.~* subje...,NaN,NaN,t,f,t,case_control prospective,body weight changes; body weight; weight loss...
2,NCT03789825,NaN,2018-11-14,NaN,NaN,2025-01-16,2018-12-28,2018-12-31,ACTUAL,NaN,...,t,General population above 40 years,inclusion criteria:~* age ≥ 40 years~* able to...,NaN,NaN,t,f,t,other prospective,fibrosis; pathologic processes; digestive sys...
3,NCT02438839,NaN,2015-05-05,NaN,NaN,2025-01-02,2015-05-05,2015-05-08,ESTIMATED,NaN,...,f,Patients with low rectal cancer (tumor located...,inclusion criteria:~* histopathologically veri...,NaN,NaN,t,f,t,case_only prospective,colorectal neoplasms; intestinal neoplasms; g...
4,NCT06331429,NaN,2024-03-19,NaN,NaN,2024-12-03,2024-03-19,2024-03-26,ACTUAL,NaN,...,f,NaN,inclusion criteria:~* diagnosis of diabetes me...,NaN,NaN,f,t,f,non_randomized parallel prevention none,diabetes mellitus; glucose metabolism disorde...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514641,NCT05773911,NaN,2022-09-15,NaN,NaN,2023-06-22,2023-03-06,2023-03-17,ACTUAL,NaN,...,t,NaN,inclusion criteria:~patients with at least 3 t...,NaN,NaN,t,f,t,randomized parallel treatment double,pathologic processes; mouth diseases; stomato...
514642,NCT05838690,NaN,2023-03-14,NaN,NaN,2024-03-25,2023-04-21,2023-05-01,ACTUAL,NaN,...,f,NaN,inclusion criteria:~infants undergoing trachea...,NaN,NaN,f,t,f,randomized crossover prevention none,randomized crossover prevention none incl...
514643,NCT01531101,NaN,2012-02-08,NaN,NaN,2020-03-16,2012-02-09,2012-02-10,ESTIMATED,NaN,...,f,NaN,inclusion criteria:~* adolescents aged 16 - 18...,NaN,NaN,t,t,f,randomized parallel treatment triple,behavioral symptoms; depression randomized p...
514644,NCT03506659,NaN,2018-03-15,NaN,NaN,2020-02-26,2018-04-23,2018-04-24,ACTUAL,NaN,...,t,NaN,inclusion criteria:~* adult~exclusion criteria...,NaN,NaN,t,f,t,non_randomized parallel diagnostic none,non_randomized parallel diagnostic none i...


In [11]:
ctod_labels = pd.concat([CTO_phase1_preds, CTO_phase2_preds, CTO_phase3_preds], ignore_index=True)
ctod_data = pd.merge(ctod_labels, studies_with_features, on='nct_id', how='left')
ctod_data.dropna(subset=['features'], inplace=True)

In [12]:
ctod_data

,nct_id,label,nlm_download_date_description,study_first_submitted_date,results_first_submitted_date,disposition_first_submitted_date,last_update_submitted_date,study_first_submitted_qc_date,study_first_posted_date,study_first_posted_date_type,...,healthy_volunteers,population,criteria,gender_description,gender_based,adult,child,older_adult,design,features
1,NCT03592264,0,NaN,2018-05-21,NaN,NaN,2024-10-11,2018-07-09,2018-07-19,ACTUAL,...,f,NaN,inclusion criteria (all subjects):~1. at least...,NaN,NaN,t,f,t,non_randomized sequential treatment none,"PHASE1/PHASE2 neoplasms; carcinoma; neoplasms,..."
2,NCT00811083,1,NaN,2008-12-17,NaN,NaN,2008-12-17,2008-12-17,2008-12-18,ESTIMATED,...,f,NaN,inclusion criteria:~phase one~1. children with...,NaN,NaN,f,t,f,randomized parallel treatment triple,PHASE1/PHASE2 autism spectrum disorder; child ...
3,NCT05988021,1,NaN,2023-08-04,NaN,NaN,2023-08-04,2023-08-04,2023-08-14,ACTUAL,...,t,NaN,inclusion criteria:~* male or female participa...,NaN,NaN,t,f,f,non_randomized sequential basic_science none,PHASE1 vascular diseases; cardiovascular disea...
4,NCT00001125,1,NaN,2000-01-17,NaN,NaN,2021-10-28,2001-08-30,2001-08-31,ESTIMATED,...,f,NaN,inclusion criteria~children may be eligible fo...,NaN,NaN,t,t,f,prevention none,PHASE1 infections; virus diseases; varicella z...
5,NCT02529553,1,NaN,2015-08-19,2020-03-08,NaN,2020-04-09,2015-08-19,2015-08-20,ESTIMATED,...,f,NaN,inclusion criteria:~* have advanced or metasta...,NaN,NaN,t,f,t,single_group treatment none,PHASE1 neoplastic processes; pathologic proces...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131155,NCT02023879,1,NaN,2013-12-06,2017-01-24,2015-10-08,2018-06-29,2013-12-24,2013-12-30,ESTIMATED,...,f,NaN,inclusion criteria:~participants with primary ...,NaN,NaN,t,f,t,randomized parallel treatment triple,PHASE3 hyperlipidemias; dyslipidemias; lipid m...
131156,NCT02023879,1,NaN,2013-12-06,2017-01-24,2015-10-08,2018-06-29,2013-12-24,2013-12-30,ESTIMATED,...,f,NaN,inclusion criteria:~participants with primary ...,NaN,NaN,t,f,t,randomized parallel treatment triple,PHASE3 hyperlipidemias; dyslipidemias; lipid m...
131157,NCT02023879,1,NaN,2013-12-06,2017-01-24,2015-10-08,2018-06-29,2013-12-24,2013-12-30,ESTIMATED,...,f,NaN,inclusion criteria:~participants with primary ...,NaN,NaN,t,f,t,randomized parallel treatment triple,PHASE3 hyperlipidemias; dyslipidemias; lipid m...
131158,NCT01214382,1,NaN,2010-09-30,NaN,NaN,2016-04-04,2010-10-01,2010-10-05,ESTIMATED,...,f,NaN,inclusion criteria:~* 1. male or female betwee...,NaN,NaN,t,f,f,single_group none,PHASE3 antidepressive agents; membrane transp...


In [13]:
ctod_data.to_csv("ctod_data.csv", index=False)
print("done")

done


In [4]:
# drugbank_df = pd.read_csv("./Data/drugbank_drugs_info.csv")

# drugbank_clean = drugbank_df[['trial_id', 'name', 'description.1', 'moldb_smiles']].copy()
# drugbank_clean.columns = ['nct_id', 'drug', 'description', 'smiles']
# drugbank_clean.dropna(subset=['smiles'], inplace=True)

# # Agrupar por ensaio
# drugbank_grouped = drugbank_clean.groupby('nct_id').agg({ 
#     'drug': lambda x: '; '.join(sorted(set(x))),
#     'description': lambda x: '; '.join(sorted(set(x.dropna()))),
#     'smiles': lambda x: '; '.join(sorted(set(x.dropna())))
# }).reset_index()

C:\Users\Asus\AppData\Local\Temp\ipykernel_16388\1475764159.py:1: DtypeWarning: Columns (30) have mixed types. Specify dtype option on import or set low_memory=False.
  drugbank_df = pd.read_csv("./Data/drugbank_drugs_info.csv")


In [5]:
# ctod_merged = pd.merge(ctod_data, drugbank_grouped, on='nct_id', how='left')

In [6]:
# ctod_final = ctod_merged[[
#     'nct_id', 'phase', 'diseases', 'drugs', 'smiles',
#     'description', 'criteria', 'label', 'features'
# ]]

In [14]:
import pandas as pd
ctod_final = pd.read_csv("Data/drugbank_info_smiles.csv")
ctod_final

C:\Users\Carol\AppData\Local\Temp\ipykernel_15164\3262368857.py:2: DtypeWarning: Columns (63,79,80,92,93) have mixed types. Specify dtype option on import or set low_memory=False.
  ctod_final = pd.read_csv("Data/drugbank_info_smiles.csv")


,nct_id,drug_name,smiles,description,iupac,molecular_weight,synonyms,max_phase,drug_type,mechanism,...,healthy_volunteers,population,criteria,gender_description,gender_based,adult,child,older_adult,design,features
0,NCT00000114,vitamin e,CC1=C(C2=C(CCC(O2)(C)CCCC(C)CCCC(C)CCCC(C)C)C(...,NaN,"(2R)-2,5,7,8-tetramethyl-2-[(4R,8R)-4,8,12-tri...",430.7,VITAMIN E; alpha-Tocopherol; 59-02-9; D-alpha-...,4.0,Unknown,NaN,...,NaN,NaN,men and nonpregnant women between ages 18 and ...,NaN,NaN,t,f,f,randomized factorial treatment double,PHASE3 retinal diseases; eye diseases; eye dis...
1,NCT00000115,acetazolamide,CC(=O)NC1=NN=C(S1)S(=O)(=O)N,Subjects will begin with four 250 mg tablets d...,"N-(5-sulfamoyl-1,3,4-thiadiazol-2-yl)acetamide",222.3,acetazolamide; 59-66-5; Diamox; Acetamox; Neph...,4.0,Small molecule,Carbonic anhydrase I inhibitor,...,NaN,NaN,males and females 8 years of age or older and ...,NaN,NaN,t,t,t,randomized crossover treatment double,PHASE2 macular degeneration; retinal degenerat...
2,NCT00000122,fluorouracil,C1=C(C(=O)NC(=O)N1)F,NaN,"5-fluoro-1H-pyrimidine-2,4-dione",130.08,5-Fluorouracil; fluorouracil; 51-21-8; 5-FU; F...,4.0,Small molecule,Thymidylate synthase inhibitor,...,f,NaN,men and women with uncontrolled intraocular pr...,NaN,NaN,t,t,t,randomized treatment double,PHASE3 ocular hypertension; eye diseases; glau...
3,NCT00000134,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,"60 mg/kg every 8 hours, 90 mg/kg/day","2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,...,f,NaN,inclusion criteria: males and females eligible...,NaN,NaN,t,f,t,randomized factorial treatment double,PHASE3 blood-borne infections; communicable di...
4,NCT00000136,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,"60 mg/kg every 8 hours, 90 mg/kg/day","2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,...,f,NaN,inclusion criteria:~* cmv retinitis in one or ...,NaN,NaN,t,t,t,randomized parallel treatment single,PHASE3 infections; virus diseases; retinal dis...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61519,NCT06649409,eplerenone; fludrocortisone,CC12CCC(=O)C=C1CC(C3C24C(O4)CC5(C3CCC56CCC(=O)...,Fludrocortisone 0.2 mg by mouth daily for 30 d...,"(8S,9R,10S,11S,13S,14S,17R)-9-fluoro-11,17-dih...",380.4; 414.5,Eplerenone; Inspra; Epoxymexrenone; 107724-20-...,2.0; 4.0,Small molecule,Mineralocorticoid receptor antagonist,...,t,NaN,inclusion criteria:~1. age of 18 to 55 years i...,NaN,NaN,t,f,f,randomized parallel other none,PHASE1 anti-inflammatory agents; antihyperten...
61520,NCT06654531,dexmedetomidine,CC1=C(C(=CC=C1)C(C)C2=CN=CN2)C,NaN,"5-[(1S)-1-(2,3-dimethylphenyl)ethyl]-1H-imidazole",200.28,DEXMEDETOMIDINE; 113775-47-6; Dexmedetomidina;...,4.0,Small molecule,NaN,...,f,NaN,inclusion criteria:~* asa physical status i-ii...,NaN,NaN,t,f,f,randomized parallel prevention none,PHASE2/PHASE3 adrenergic agents; adrenergic a...
61521,NCT06654570,letrozole,C1=CC(=CC=C1C#N)C(C2=CC=C(C=C2)C#N)N3C=NC=N3,Letrozole 2.5 mg by mouth daily for 24 weeks,"4-[(4-cyanophenyl)-(1,2,4-triazol-1-yl)methyl]...",285.3,"letrozole; 112809-51-5; Femara; 4,4'-((1h-1,2,...",4.0,Small molecule,Cytochrome P450 19A1 inhibitor,...,f,NaN,inclusion criteria:~* eligible patients were p...,Females,t,t,f,t,single_group treatment none,PHASE2 neoplasms by site; neoplasms; breast di...
61522,NCT06655818,dostarlimab,NaN,NaN,NaN,NaN,NaN,4.0,Antibody,Programmed cell death protein 1 antagonist,...,f,NaN,inclusion criteria:~* participant must be 18 y...,NaN,NaN,t,f,t,single_group treatment none,PHASE1/PHASE2 neoplasms by histologic type; ne...


In [15]:
import ast

disease_icd_df = pd.read_csv("data/diseases.csv")
disease_icd_df['disease'] = disease_icd_df['disease'].str.lower()

# Convert string-repr lists to real lists
def parse_icd(x):
    try:
        return ast.literal_eval(x)
    except:
        return ['None']

disease_icd_df['icd'] = disease_icd_df['icd'].apply(parse_icd)

def filter_icd_list(icd_list):
    cleaned = []
    for code in icd_list:
        code = code.strip()
        if not code or code == 'None':
            continue
        if code.startswith('O'):  # obstetrics
            continue
        if code.startswith('Z'):  # general or encounter codes
            continue
        if code.endswith('9'):   # often unspecified
            continue
        cleaned.append(code)
    return sorted(set(cleaned))

# Prepare diseases in ctod_final
ctod_final['diseases_clean'] = ctod_final['diseases'].str.lower()

In [16]:
ctod_final

,nct_id,drug_name,smiles,description,iupac,molecular_weight,synonyms,max_phase,drug_type,mechanism,...,population,criteria,gender_description,gender_based,adult,child,older_adult,design,features,diseases_clean
0,NCT00000114,vitamin e,CC1=C(C2=C(CCC(O2)(C)CCCC(C)CCCC(C)CCCC(C)C)C(...,NaN,"(2R)-2,5,7,8-tetramethyl-2-[(4R,8R)-4,8,12-tri...",430.7,VITAMIN E; alpha-Tocopherol; 59-02-9; D-alpha-...,4.0,Unknown,NaN,...,NaN,men and nonpregnant women between ages 18 and ...,NaN,NaN,t,f,f,randomized factorial treatment double,PHASE3 retinal diseases; eye diseases; eye dis...,"retinal diseases; eye diseases; eye diseases, ..."
1,NCT00000115,acetazolamide,CC(=O)NC1=NN=C(S1)S(=O)(=O)N,Subjects will begin with four 250 mg tablets d...,"N-(5-sulfamoyl-1,3,4-thiadiazol-2-yl)acetamide",222.3,acetazolamide; 59-66-5; Diamox; Acetamox; Neph...,4.0,Small molecule,Carbonic anhydrase I inhibitor,...,NaN,males and females 8 years of age or older and ...,NaN,NaN,t,t,t,randomized crossover treatment double,PHASE2 macular degeneration; retinal degenerat...,macular degeneration; retinal degeneration; re...
2,NCT00000122,fluorouracil,C1=C(C(=O)NC(=O)N1)F,NaN,"5-fluoro-1H-pyrimidine-2,4-dione",130.08,5-Fluorouracil; fluorouracil; 51-21-8; 5-FU; F...,4.0,Small molecule,Thymidylate synthase inhibitor,...,NaN,men and women with uncontrolled intraocular pr...,NaN,NaN,t,t,t,randomized treatment double,PHASE3 ocular hypertension; eye diseases; glau...,ocular hypertension; eye diseases; glaucoma
3,NCT00000134,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,"60 mg/kg every 8 hours, 90 mg/kg/day","2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,...,NaN,inclusion criteria: males and females eligible...,NaN,NaN,t,f,t,randomized factorial treatment double,PHASE3 blood-borne infections; communicable di...,blood-borne infections; communicable diseases;...
4,NCT00000136,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,"60 mg/kg every 8 hours, 90 mg/kg/day","2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,...,NaN,inclusion criteria:~* cmv retinitis in one or ...,NaN,NaN,t,t,t,randomized parallel treatment single,PHASE3 infections; virus diseases; retinal dis...,infections; virus diseases; retinal diseases; ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61519,NCT06649409,eplerenone; fludrocortisone,CC12CCC(=O)C=C1CC(C3C24C(O4)CC5(C3CCC56CCC(=O)...,Fludrocortisone 0.2 mg by mouth daily for 30 d...,"(8S,9R,10S,11S,13S,14S,17R)-9-fluoro-11,17-dih...",380.4; 414.5,Eplerenone; Inspra; Epoxymexrenone; 107724-20-...,2.0; 4.0,Small molecule,Mineralocorticoid receptor antagonist,...,NaN,inclusion criteria:~1. age of 18 to 55 years i...,NaN,NaN,t,f,f,randomized parallel other none,PHASE1 anti-inflammatory agents; antihyperten...,NaN
61520,NCT06654531,dexmedetomidine,CC1=C(C(=CC=C1)C(C)C2=CN=CN2)C,NaN,"5-[(1S)-1-(2,3-dimethylphenyl)ethyl]-1H-imidazole",200.28,DEXMEDETOMIDINE; 113775-47-6; Dexmedetomidina;...,4.0,Small molecule,NaN,...,NaN,inclusion criteria:~* asa physical status i-ii...,NaN,NaN,t,f,f,randomized parallel prevention none,PHASE2/PHASE3 adrenergic agents; adrenergic a...,NaN
61521,NCT06654570,letrozole,C1=CC(=CC=C1C#N)C(C2=CC=C(C=C2)C#N)N3C=NC=N3,Letrozole 2.5 mg by mouth daily for 24 weeks,"4-[(4-cyanophenyl)-(1,2,4-triazol-1-yl)methyl]...",285.3,"letrozole; 112809-51-5; Femara; 4,4'-((1h-1,2,...",4.0,Small molecule,Cytochrome P450 19A1 inhibitor,...,NaN,inclusion criteria:~* eligible patients were p...,Females,t,t,f,t,single_group treatment none,PHASE2 neoplasms by site; neoplasms; breast di...,neoplasms by site; neoplasms; breast diseases;...
61522,NCT06655818,dostarlimab,NaN,NaN,NaN,NaN,NaN,4.0,Antibody,Programmed 

In [17]:
def map_icd_codes(disease_text):
    if pd.isna(disease_text):
        return '[]'
    
    found_icds = set()
    for disease in disease_text.split(';'):
        disease = disease.strip()
        match = disease_icd_df[disease_icd_df['disease'] == disease]
        if not match.empty:
            for icd_list in match['icd']:
                found_icds.update(icd_list)
    
    filtered = filter_icd_list(found_icds)
    return str(filtered)


In [18]:
ctod_final['icdcodes'] = ctod_final['diseases_clean'].apply(map_icd_codes)
ctod_final.drop(columns=['diseases_clean'], inplace=True)

In [19]:
ctod_final

,nct_id,drug_name,smiles,description,iupac,molecular_weight,synonyms,max_phase,drug_type,mechanism,...,population,criteria,gender_description,gender_based,adult,child,older_adult,design,features,icdcodes
0,NCT00000114,vitamin e,CC1=C(C2=C(CCC(O2)(C)CCCC(C)CCCC(C)CCCC(C)C)C(...,NaN,"(2R)-2,5,7,8-tetramethyl-2-[(4R,8R)-4,8,12-tri...",430.7,VITAMIN E; alpha-Tocopherol; 59-02-9; D-alpha-...,4.0,Unknown,NaN,...,NaN,men and nonpregnant women between ages 18 and ...,NaN,NaN,t,f,f,randomized factorial treatment double,PHASE3 retinal diseases; eye diseases; eye dis...,"['H35.40', 'H35.54']"
1,NCT00000115,acetazolamide,CC(=O)NC1=NN=C(S1)S(=O)(=O)N,Subjects will begin with four 250 mg tablets d...,"N-(5-sulfamoyl-1,3,4-thiadiazol-2-yl)acetamide",222.3,acetazolamide; 59-66-5; Diamox; Acetamox; Neph...,4.0,Small molecule,Carbonic anhydrase I inhibitor,...,NaN,males and females 8 years of age or older and ...,NaN,NaN,t,t,t,randomized crossover treatment double,PHASE2 macular degeneration; retinal degenerat...,"['H34.8110', 'H34.8120', 'H34.8130', 'H35.30',..."
2,NCT00000122,fluorouracil,C1=C(C(=O)NC(=O)N1)F,NaN,"5-fluoro-1H-pyrimidine-2,4-dione",130.08,5-Fluorouracil; fluorouracil; 51-21-8; 5-FU; F...,4.0,Small molecule,Thymidylate synthase inhibitor,...,NaN,men and women with uncontrolled intraocular pr...,NaN,NaN,t,t,t,randomized treatment double,PHASE3 ocular hypertension; eye diseases; glau...,"['B73.02', 'H40.051', 'H40.052', 'H40.053', 'H..."
3,NCT00000134,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,"60 mg/kg every 8 hours, 90 mg/kg/day","2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,...,NaN,inclusion criteria: males and females eligible...,NaN,NaN,t,f,t,randomized factorial treatment double,PHASE3 blood-borne infections; communicable di...,"['A31.8', 'A63.8', 'A69.1', 'A70', 'B97.33', '..."
4,NCT00000136,foscarnet; ganciclovir,C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,"60 mg/kg every 8 hours, 90 mg/kg/day","2-amino-9-(1,3-dihydroxypropan-2-yloxymethyl)-...",126.01; 255.23,foscarnet; Phosphonoformic acid; Phosphonoform...,4.0,Small molecule,Human herpesvirus 1 DNA polymerase inhibitor,...,NaN,inclusion criteria:~* cmv retinitis in one or ...,NaN,NaN,t,t,t,randomized parallel treatment single,PHASE3 infections; virus diseases; retinal dis...,"['A31.8', 'A69.1', 'A70', 'B97.33', 'B97.34', ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61519,NCT06649409,eplerenone; fludrocortisone,CC12CCC(=O)C=C1CC(C3C24C(O4)CC5(C3CCC56CCC(=O)...,Fludrocortisone 0.2 mg by mouth daily for 30 d...,"(8S,9R,10S,11S,13S,14S,17R)-9-fluoro-11,17-dih...",380.4; 414.5,Eplerenone; Inspra; Epoxymexrenone; 107724-20-...,2.0; 4.0,Small molecule,Mineralocorticoid receptor antagonist,...,NaN,inclusion criteria:~1. age of 18 to 55 years i...,NaN,NaN,t,f,f,randomized parallel other none,PHASE1 anti-inflammatory agents; antihyperten...,[]
61520,NCT06654531,dexmedetomidine,CC1=C(C(=CC=C1)C(C)C2=CN=CN2)C,NaN,"5-[(1S)-1-(2,3-dimethylphenyl)ethyl]-1H-imidazole",200.28,DEXMEDETOMIDINE; 113775-47-6; Dexmedetomidina;...,4.0,Small molecule,NaN,...,NaN,inclusion criteria:~* asa physical status i-ii...,NaN,NaN,t,f,f,randomized parallel prevention none,PHASE2/PHASE3 adrenergic agents; adrenergic a...,[]
61521,NCT06654570,letrozole,C1=CC(=CC=C1C#N)C(C2=CC=C(C=C2)C#N)N3C=NC=N3,Letrozole 2.5 mg by mouth daily for 24 weeks,"4-[(4-cyanophenyl)-(1,2,4-triazol-1-yl)methyl]...",285.3,"letrozole; 112809-51-5; Femara; 4,4'-((1h-1,2,...",4.0,Small molecule,Cytochrome P450 19A1 inhibitor,...,NaN,inclusion criteria:~* eligible patients were p...,Females,t,t,f,t,single_group treatment none,PHASE2 neoplasms by site; neoplasms; breast di...,"['C96.Z', 'H31.23', 'H47.20', 'H47.22', 'N50.0..."
61522,NCT06655818,dostarlimab,NaN,NaN,NaN,NaN,NaN,4.0,Antibody,Programmed cell death protein 1 

In [20]:
ctod_final.columns

Index(['nct_id', 'drug_name', 'smiles', 'description', 'iupac',
       'molecular_weight', 'synonyms', 'max_phase', 'drug_type', 'mechanism',
       'target', 'label', 'nlm_download_date_description',
       'study_first_submitted_date', 'results_first_submitted_date',
       'disposition_first_submitted_date', 'last_update_submitted_date',
       'study_first_submitted_qc_date', 'study_first_posted_date',
       'study_first_posted_date_type', 'results_first_submitted_qc_date',
       'results_first_posted_date', 'results_first_posted_date_type',
       'disposition_first_submitted_qc_date', 'disposition_first_posted_date',
       'disposition_first_posted_date_type', 'last_update_submitted_qc_date',
       'last_update_posted_date', 'last_update_posted_date_type',
       'start_month_year', 'start_date_type', 'start_date',
       'verification_month_year', 'verification_date', 'completion_month_year',
       'completion_date_type', 'completion_date',
       'primary_completion_month_

In [22]:
selected_columns = [
    'nct_id',
    'brief_title',
    'overall_status',
    'start_date',
    'completion_date',
    'phase',
    'enrollment',
    'diseases',
    'drug_name',
    'drug_type',
    'description',
    'smiles',
    'criteria',
    'label',
    'icdcodes'
]

ctod_final_filtered = ctod_final[selected_columns].copy()
ctod_final_filtered = ctod_final_filtered.rename(columns={
    'drug_name': 'drugs'
})
ctod_final_filtered.head()

,nct_id,brief_title,overall_status,start_date,completion_date,phase,enrollment,diseases,drugs,drug_type,description,smiles,criteria,label,icdcodes
0,NCT00000114,Randomized Trial of Vitamin A and Vitamin E Su...,COMPLETED,1984-05-31,1987-06-30,PHASE3,NaN,"retinal diseases; eye diseases; eye diseases, ...",vitamin e,Unknown,NaN,CC1=C(C2=C(CCC(O2)(C)CCCC(C)CCCC(C)CCCC(C)C)C(...,men and nonpregnant women between ages 18 and ...,1,"['H35.40', 'H35.54']"
1,NCT00000115,Randomized Trial of Acetazolamide for Uveitis-...,COMPLETED,1990-12-31,1994-06-30,PHASE2,NaN,macular degeneration; retinal degeneration; re...,acetazolamide,Small molecule,Subjects will begin with four 250 mg tablets d...,CC(=O)NC1=NN=C(S1)S(=O)(=O)N,males and females 8 years of age or older and ...,1,"['H34.8110', 'H34.8120', 'H34.8130', 'H35.30',..."
2,NCT00000122,Fluorouracil Filtering Surgery Study (FFSS),COMPLETED,1985-09-30,1988-06-30,PHASE3,NaN,ocular hypertension; eye diseases; glaucoma,fluorouracil,Small molecule,NaN,C1=C(C(=O)NC(=O)N1)F,men and women with uncontrolled intraocular pr...,1,"['B73.02', 'H40.051', 'H40.052', 'H40.053', 'H..."
3,NCT00000134,Studies of the Ocular Complications of AIDS (S...,COMPLETED,1992-12-31,1995-03-31,PHASE3,279.0,blood-borne infections; communicable diseases;...,foscarnet; ganciclovir,Small molecule,"60 mg/kg every 8 hours, 90 mg/kg/day",C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,inclusion criteria: males and females eligible...,1,"['A31.8', 'A63.8', 'A69.1', 'A70', 'B97.33', '..."
4,NCT00000136,Studies of the Ocular Complications of AIDS (S...,COMPLETED,1990-03-31,1991-10-31,PHASE3,234.0,infections; virus diseases; retinal diseases; ...,foscarnet; ganciclovir,Small molecule,"60 mg/kg every 8 hours, 90 mg/kg/day",C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,inclusion criteria:~* cmv retinitis in one or ...,1,"['A31.8', 'A69.1', 'A70', 'B97.33', 'B97.34', ..."


In [23]:
ctod_final_filtered["overall_status"].unique()

array(['COMPLETED', 'TERMINATED', 'WITHDRAWN', 'SUSPENDED', 'RECRUITING',
       'ACTIVE_NOT_RECRUITING', 'NOT_YET_RECRUITING'], dtype=object)

In [24]:
selected_statuses = ["COMPLETED", "TERMINATED", "SUSPENDED", "WITHDRAWN"]
completed_trials = ctod_final_filtered[ctod_final_filtered["overall_status"].isin(selected_statuses)]

In [25]:
completed_trials

,nct_id,brief_title,overall_status,start_date,completion_date,phase,enrollment,diseases,drugs,drug_type,description,smiles,criteria,label,icdcodes
0,NCT00000114,Randomized Trial of Vitamin A and Vitamin E Su...,COMPLETED,1984-05-31,1987-06-30,PHASE3,NaN,"retinal diseases; eye diseases; eye diseases, ...",vitamin e,Unknown,NaN,CC1=C(C2=C(CCC(O2)(C)CCCC(C)CCCC(C)CCCC(C)C)C(...,men and nonpregnant women between ages 18 and ...,1,"['H35.40', 'H35.54']"
1,NCT00000115,Randomized Trial of Acetazolamide for Uveitis-...,COMPLETED,1990-12-31,1994-06-30,PHASE2,NaN,macular degeneration; retinal degeneration; re...,acetazolamide,Small molecule,Subjects will begin with four 250 mg tablets d...,CC(=O)NC1=NN=C(S1)S(=O)(=O)N,males and females 8 years of age or older and ...,1,"['H34.8110', 'H34.8120', 'H34.8130', 'H35.30',..."
2,NCT00000122,Fluorouracil Filtering Surgery Study (FFSS),COMPLETED,1985-09-30,1988-06-30,PHASE3,NaN,ocular hypertension; eye diseases; glaucoma,fluorouracil,Small molecule,NaN,C1=C(C(=O)NC(=O)N1)F,men and women with uncontrolled intraocular pr...,1,"['B73.02', 'H40.051', 'H40.052', 'H40.053', 'H..."
3,NCT00000134,Studies of the Ocular Complications of AIDS (S...,COMPLETED,1992-12-31,1995-03-31,PHASE3,279.0,blood-borne infections; communicable diseases;...,foscarnet; ganciclovir,Small molecule,"60 mg/kg every 8 hours, 90 mg/kg/day",C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,inclusion criteria: males and females eligible...,1,"['A31.8', 'A63.8', 'A69.1', 'A70', 'B97.33', '..."
4,NCT00000136,Studies of the Ocular Complications of AIDS (S...,COMPLETED,1990-03-31,1991-10-31,PHASE3,234.0,infections; virus diseases; retinal diseases; ...,foscarnet; ganciclovir,Small molecule,"60 mg/kg every 8 hours, 90 mg/kg/day",C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,inclusion criteria:~* cmv retinitis in one or ...,1,"['A31.8', 'A69.1', 'A70', 'B97.33', 'B97.34', ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61519,NCT06649409,Evaluation of Vamorolone Mineralocorticoid Rec...,COMPLETED,2024-06-17,2024-07-08,PHASE1,30.0,NaN,eplerenone; fludrocortisone,Small molecule,Fludrocortisone 0.2 mg by mouth daily for 30 d...,CC12CCC(=O)C=C1CC(C3C24C(O4)CC5(C3CCC56CCC(=O)...,inclusion criteria:~1. age of 18 to 55 years i...,1,[]
61520,NCT06654531,Effect of Intravenous or Intrathecal Dexmedeto...,COMPLETED,2022-01-02,2024-03-10,PHASE2/PHASE3,50.0,NaN,dexmedetomidine,Small molecule,NaN,CC1=C(C(=CC=C1)C(C)C2=CN=CN2)C,inclusion criteria:~* asa physical status i-ii...,1,[]
61521,NCT06654570,Effects of Vaginal Estrogen on Serum Estradiol...,COMPLETED,2020-04-08,2024-10-10,PHASE2,20.0,neoplasms by site; neoplasms; breast diseases;...,letrozole,Small molecule,Letrozole 2.5 mg by mouth daily for 24 weeks,C1=CC(=CC=C1C#N)C(C2=CC=C(C=C2)C#N)N3C=NC=N3,inclusion criteria:~* eligible patients were p...,1,"['C96.Z', 'H31.23', 'H47.20', 'H47.22', 'N50.0..."
61522,NCT06655818,Sub-study of Belantamab Mafodotin (GSK2857916)...,TERMINATED,2021-03-09,2024-02-14,PHASE1/PHASE2,4.0,neoplasms by histologic type; neoplasms; hemos...,dostarlimab,Antibody,NaN,NaN,inclusion criteria:~* participant must be 18 y...,0,"['C90.00', 'C90.01', 'C90.02', 'C96.Z', 'G46.8']"


In [12]:
completed_trials["drug_type"].unique()

array(['Unknown', 'Small molecule', 'Small molecule; Unknown', 'Protein',
       'Oligosaccharide; Unknown', 'Enzyme; Small molecule', 'Enzyme',
       'Protein; Small molecule', 'Protein; Small molecule; Unknown',
       'Antibody', 'Antibody; Small molecule',
       'Oligosaccharide; Small molecule; Unknown',
       'Enzyme; Small molecule; Unknown', 'Antibody; Protein',
       'Enzyme; Protein; Small molecule',
       'Enzyme; Protein; Small molecule; Unknown',
       'Oligonucleotide; Small molecule', 'Protein; Unknown', nan,
       'Antibody; Protein; Small molecule', 'Oligonucleotide',
       'Antibody; Small molecule; Unknown', 'Antibody; Oligonucleotide',
       'Antibody; Enzyme; Small molecule', 'Oligosaccharide',
       'Antibody; Enzyme', 'Oligonucleotide; Small molecule; Unknown',
       'Antibody drug conjugate',
       'Antibody; Enzyme; Small molecule; Unknown',
       'Oligosaccharide; Small molecule',
       'Oligonucleotide; Protein; Small molecule',
       'Antibody

In [26]:
exclude = ["Unknown", "Oligosaccharide"]

df = completed_trials.dropna(subset = ["drug_type"])

clean_df = df[~df['drug_type'].apply(lambda x: any(k in x for k in exclude))]
clean_df

,nct_id,brief_title,overall_status,start_date,completion_date,phase,enrollment,diseases,drugs,drug_type,description,smiles,criteria,label,icdcodes
1,NCT00000115,Randomized Trial of Acetazolamide for Uveitis-...,COMPLETED,1990-12-31,1994-06-30,PHASE2,NaN,macular degeneration; retinal degeneration; re...,acetazolamide,Small molecule,Subjects will begin with four 250 mg tablets d...,CC(=O)NC1=NN=C(S1)S(=O)(=O)N,males and females 8 years of age or older and ...,1,"['H34.8110', 'H34.8120', 'H34.8130', 'H35.30',..."
2,NCT00000122,Fluorouracil Filtering Surgery Study (FFSS),COMPLETED,1985-09-30,1988-06-30,PHASE3,NaN,ocular hypertension; eye diseases; glaucoma,fluorouracil,Small molecule,NaN,C1=C(C(=O)NC(=O)N1)F,men and women with uncontrolled intraocular pr...,1,"['B73.02', 'H40.051', 'H40.052', 'H40.053', 'H..."
3,NCT00000134,Studies of the Ocular Complications of AIDS (S...,COMPLETED,1992-12-31,1995-03-31,PHASE3,279.0,blood-borne infections; communicable diseases;...,foscarnet; ganciclovir,Small molecule,"60 mg/kg every 8 hours, 90 mg/kg/day",C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,inclusion criteria: males and females eligible...,1,"['A31.8', 'A63.8', 'A69.1', 'A70', 'B97.33', '..."
4,NCT00000136,Studies of the Ocular Complications of AIDS (S...,COMPLETED,1990-03-31,1991-10-31,PHASE3,234.0,infections; virus diseases; retinal diseases; ...,foscarnet; ganciclovir,Small molecule,"60 mg/kg every 8 hours, 90 mg/kg/day",C(=O)(O)P(=O)(O)O; C1=NC2=C(N1COC(CO)CO)N=C(NC...,inclusion criteria:~* cmv retinitis in one or ...,1,"['A31.8', 'A69.1', 'A70', 'B97.33', 'B97.34', ..."
5,NCT00000142,Studies of the Ocular Complications of AIDS (S...,COMPLETED,1994-04-30,1996-02-29,PHASE2/PHASE3,64.0,infections; virus diseases; retinal diseases; ...,cidofovir,Small molecule,NaN,C1=CN(C(=O)N=C1N)CC(CO)OCP(=O)(O)O,inclusion criteria:~* diagnosis of aids accord...,1,"['A31.8', 'A69.1', 'A70', 'B97.33', 'B97.34', ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61519,NCT06649409,Evaluation of Vamorolone Mineralocorticoid Rec...,COMPLETED,2024-06-17,2024-07-08,PHASE1,30.0,NaN,eplerenone; fludrocortisone,Small molecule,Fludrocortisone 0.2 mg by mouth daily for 30 d...,CC12CCC(=O)C=C1CC(C3C24C(O4)CC5(C3CCC56CCC(=O)...,inclusion criteria:~1. age of 18 to 55 years i...,1,[]
61520,NCT06654531,Effect of Intravenous or Intrathecal Dexmedeto...,COMPLETED,2022-01-02,2024-03-10,PHASE2/PHASE3,50.0,NaN,dexmedetomidine,Small molecule,NaN,CC1=C(C(=CC=C1)C(C)C2=CN=CN2)C,inclusion criteria:~* asa physical status i-ii...,1,[]
61521,NCT06654570,Effects of Vaginal Estrogen on Serum Estradiol...,COMPLETED,2020-04-08,2024-10-10,PHASE2,20.0,neoplasms by site; neoplasms; breast diseases;...,letrozole,Small molecule,Letrozole 2.5 mg by mouth daily for 24 weeks,C1=CC(=CC=C1C#N)C(C2=CC=C(C=C2)C#N)N3C=NC=N3,inclusion criteria:~* eligible patients were p...,1,"['C96.Z', 'H31.23', 'H47.20', 'H47.22', 'N50.0..."
61522,NCT06655818,Sub-study of Belantamab Mafodotin (GSK2857916)...,TERMINATED,2021-03-09,2024-02-14,PHASE1/PHASE2,4.0,neoplasms by histologic type; neoplasms; hemos...,dostarlimab,Antibody,NaN,NaN,inclusion criteria:~* participant must be 18 y...,0,"['C90.00', 'C90.01', 'C90.02', 'C96.Z', 'G46.8']"


In [14]:
# Optional: save it
clean_df.to_csv("CTOD_clean_dataset.csv", index=False)
print("Added ICD codes and saved to ctod_complete_dataset.csv")

Added ICD codes and saved to ctod_complete_dataset.csv
